# Travel Destination Classification - Multi-Feature Evaluation
## Predicting Country and Time of Day from Visual Features

**Authors:** Machine Learning Assignment 3  
**Date:** January 2026

In [220]:
## 1. Setup and Imports

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json
import seaborn as sns
from pathlib import Path
import os
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

# Try to import PyTorch
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
    print("✓ PyTorch available")
except ImportError:
    TORCH_AVAILABLE = False
    print("✗ PyTorch not available")

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Path configuration
if '__file__' in dir():
    BASE_PATH = Path(__file__).parent.parent
else:
    BASE_PATH = Path(os.getcwd())
    if BASE_PATH.name == 'notebooks':
        BASE_PATH = BASE_PATH.parent

DATA_PATH = BASE_PATH / 'data'
RESULTS_PATH = BASE_PATH / 'results' / 'metrics'
FIGURES_PATH = BASE_PATH / 'results' / 'figures'

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

print(f"Base path: {BASE_PATH}")

✓ PyTorch available
Base path: c:\Users\qadim\OneDrive - student.birzeit.edu\birzeit\5_year_1\machine_learning\assingment_3_project\travel-recommender-system


## 2. Global Configuration & Target Preparation
We prepare the targets and data split INDEPENDENTLY of the features to ensure fair comparison.

In [221]:
# 1. Load Metadata
cleaned_df = pd.read_csv(DATA_PATH / 'processed' / 'cleaned_dataset.csv')

# 2. Derive Activity
def derive_activity(text):
    text = str(text).lower()
    keywords = {
        'Nature': ['park', 'garden', 'mountain', 'lake', 'river', 'nature', 'hike', 'hiking', 'forest', 'valley', 'waterfall', 'beach', 'sea', 'ocean', 'island', 'sand', 'cave', 'rock', 'hill', 'view', 'landscape', 'sunrise', 'sunset'],
        'History': ['museum', 'castle', 'palace', 'temple', 'church', 'cathedral', 'history', 'ancient', 'monument', 'ruins', 'art', 'statue', 'shrine', 'mosque', 'tomb', 'archaeology', 'historic'],
        'Urban': ['city', 'street', 'building', 'bridge', 'tower', 'square', 'market', 'shop', 'downtown', 'urban', 'skyline', 'hotel', 'mall', 'road', 'town', 'architecture', 'skyscraper'],
    }
    for cat in ['History', 'Urban', 'Nature']:
        if any(w in text for w in keywords[cat]):
            return cat
    return 'Leisure/Other'

cleaned_df['Activity'] = cleaned_df['Description'].apply(derive_activity)

# 3. Filter Valid Data
CONFIG = {
    'top_n_countries': 15,
    'valid_times': ['Morning', 'Afternoon', 'Evening', 'Night'],
    'test_size': 0.20,
    'random_state': 42
}

valid_mask = cleaned_df['Time_of_Day_Standardized'].isin(CONFIG['valid_times'])
df = cleaned_df[valid_mask].copy().reset_index(drop=True)
valid_indices = np.where(valid_mask)[0] 

print(f"Valid samples: {len(df)}")

# 4. Encoders
country_counts_all = df['Country_Standardized'].value_counts()
top_countries = set(country_counts_all.head(CONFIG['top_n_countries']).index)
df['Country_Grouped'] = df['Country_Standardized'].apply(lambda x: x if x in top_countries else 'Other')

country_encoder = LabelEncoder()
country_encoder.fit(sorted(list(top_countries) + ['Other']))
time_encoder = LabelEncoder()
time_encoder.fit(CONFIG['valid_times'])
activity_encoder = LabelEncoder()
activity_encoder.fit(sorted(df['Activity'].unique()))

y_country = country_encoder.transform(df['Country_Grouped'].values)
y_time = time_encoder.transform(df['Time_of_Day_Standardized'].values)
y_activity = activity_encoder.transform(df['Activity'].values)

# 5. Stratified Split Schema
# Create combined stratification key
strat_key = (df['Country_Grouped'] + '_' + df['Time_of_Day_Standardized'] + '_' + df['Activity'])
strat_counts = strat_key.value_counts()
strat_key_safe = strat_key.apply(lambda x: 'OTHER' if strat_counts[x] < 2 else x)

train_idx_global, test_idx_global = train_test_split(
    list(range(len(df))), 
    test_size=CONFIG['test_size'],
    random_state=CONFIG['random_state'],
    stratify=strat_key_safe.values
)

print(f"Split prepared: {len(train_idx_global)} Train, {len(test_idx_global)} Test")

Valid samples: 945
Split prepared: 756 Train, 189 Test


## 3. Experiment Pipeline
Encapsulated function to run the full classification suite on a specific feature set.

In [222]:
def evaluate_model(y_true, y_pred, task_name):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
    }

def oversample_minority_classes(X, y, random_state=42):
    np.random.seed(random_state)
    unique_classes, counts = np.unique(y, return_counts=True)
    max_count = counts.max()
    
    X_resampled = [X]
    y_resampled = [y]
    
    for cls, count in zip(unique_classes, counts):
        if count < max_count:
            n_add = int(max_count - count)
            indices = np.where(y == cls)[0]
            if len(indices) == 0: continue
            
            add_indices = np.random.choice(indices, n_add, replace=True)
            X_add = X[add_indices]
            
            # Add slight noise (data augmentation)
            noise = np.random.normal(0, 0.05, X_add.shape)
            X_add_noisy = X_add + noise
            
            X_resampled.append(X_add_noisy)
            y_resampled.append(np.full(n_add, cls))
    
    return np.vstack(X_resampled), np.concatenate(y_resampled)

def run_experiment(feature_name, feature_file):
    print("\n" + "#" * 80)
    print(f"🚀 RUNNING EXPERIMENT: {feature_name.upper()}")
    print("#" * 80)
    
    # Setup Output Directories
    exp_fig_path = FIGURES_PATH / feature_name
    exp_fig_path.mkdir(parents=True, exist_ok=True)
    
    # 1. Load Features
    print(f"   Loading {feature_file}...")
    try:
        features_all = np.load(DATA_PATH / 'features' / feature_file)
    except FileNotFoundError:
        print(f"   ❌ File not found: {feature_file}. Skipping.")
        return None

    # Filter Valid indices (Must align with df filtering done globally)
    X_valid = features_all[valid_indices]
    
    # Split
    X_train = X_valid[train_idx_global]
    X_test = X_valid[test_idx_global]
    
    # Get Targets
    y_train_c, y_test_c = y_country[train_idx_global], y_country[test_idx_global]
    y_train_t, y_test_t = y_time[train_idx_global], y_time[test_idx_global]
    y_train_a, y_test_a = y_activity[train_idx_global], y_activity[test_idx_global]
    
    # Scale features (if not already normalized versions, but standard scaler is safe to re-apply)
    # Note: For 'simple' features (TF-IDF), StandardScaler might destroy sparsity if not careful, 
    # but here we use dense arrays so it's fine.
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Oversampling for Time Task (imbalanced)
    print("   Oversampling for Time task...")
    X_train_bal, y_train_t_bal = oversample_minority_classes(X_train_scaled, y_train_t)
    
    all_results = {}
    
    # ==========================
    # MODEL 1: KNN (k=3)
    # ==========================
    print("   Training KNN (k=3)...")
    knn3_c = KNeighborsClassifier(n_neighbors=3, metric='cosine', weights='distance')
    knn3_c.fit(X_train_scaled, y_train_c)
    
    knn3_t = KNeighborsClassifier(n_neighbors=3, metric='cosine', weights='distance')
    knn3_t.fit(X_train_scaled, y_train_t)
    
    knn3_a = KNeighborsClassifier(n_neighbors=3, metric='cosine', weights='distance')
    knn3_a.fit(X_train_scaled, y_train_a)
    
    res_knn = {
        'country': evaluate_model(y_test_c, knn3_c.predict(X_test_scaled), 'country'),
        'time': evaluate_model(y_test_t, knn3_t.predict(X_test_scaled), 'time'),
        'activity': evaluate_model(y_test_a, knn3_a.predict(X_test_scaled), 'activity')
    }
    all_results['KNN-3'] = res_knn

    # ==========================
    # MODEL 2: Random Forest
    # ==========================
    print("   Training Random Forest...")
    rf_c = RandomForestClassifier(n_estimators=200, max_depth=20, n_jobs=-1, random_state=42)
    rf_c.fit(X_train_scaled, y_train_c)
    
    rf_t = RandomForestClassifier(n_estimators=200, max_depth=20, n_jobs=-1, random_state=42)
    rf_t.fit(X_train_scaled, y_train_t) # Using standard data for RF is surprisingly effective often
    
    rf_a = RandomForestClassifier(n_estimators=200, max_depth=20, n_jobs=-1, random_state=42)
    rf_a.fit(X_train_scaled, y_train_a)
    
    res_rf = {
        'country': evaluate_model(y_test_c, rf_c.predict(X_test_scaled), 'country'),
        'time': evaluate_model(y_test_t, rf_t.predict(X_test_scaled), 'time'),
        'activity': evaluate_model(y_test_a, rf_a.predict(X_test_scaled), 'activity')
    }
    all_results['RF'] = res_rf

    # ==========================
    # MODEL 3: Neural Network
    # ==========================
    nn_res = None
    if TORCH_AVAILABLE:
        print("   Training Neural Network...")
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        class MLP(nn.Module):
            def __init__(self, in_dim, out_dim):
                super().__init__()
                self.net = nn.Sequential(
                    nn.Linear(in_dim, 512),
                    nn.BatchNorm1d(512),
                    nn.ReLU(),
                    nn.Dropout(0.4),
                    nn.Linear(512, 256),
                    nn.BatchNorm1d(256),
                    nn.ReLU(),
                    nn.Dropout(0.3),
                    nn.Linear(256, out_dim)
                )
            def forward(self, x):
                return self.net(x)

        def train_nn_task(y_train_target, y_test_target, n_classes):
            X_t = torch.FloatTensor(X_train_scaled).to(device)
            y_t = torch.LongTensor(y_train_target).to(device)
            loader = DataLoader(TensorDataset(X_t, y_t), batch_size=32, shuffle=True)
            
            model = MLP(X_train_scaled.shape[1], n_classes).to(device)
            opt = optim.AdamW(model.parameters(), lr=0.001)
            crit = nn.CrossEntropyLoss()
            
            model.train()
            for ep in range(30):
                for bx, by in loader:
                    opt.zero_grad()
                    loss = crit(model(bx), by)
                    loss.backward()
                    opt.step()
            
            model.eval()
            with torch.no_grad():
                preds = model(torch.FloatTensor(X_test_scaled).to(device)).argmax(dim=1).cpu().numpy()
            return preds

        pred_nn_c = train_nn_task(y_train_c, y_test_c, len(country_encoder.classes_))
        pred_nn_t = train_nn_task(y_train_t, y_test_t, len(time_encoder.classes_))
        pred_nn_a = train_nn_task(y_train_a, y_test_a, len(activity_encoder.classes_))
        
        nn_res = {
            'country': evaluate_model(y_test_c, pred_nn_c, 'country'),
            'time': evaluate_model(y_test_t, pred_nn_t, 'time'),
            'activity': evaluate_model(y_test_a, pred_nn_a, 'activity')
        }
        all_results['NN'] = nn_res

    # ==========================
    # VISUALIZATION & SAVING
    # ==========================
    print("   Generating Visualizations...")
    
    # 1. Bar Chart Comparison
    models_list = ['KNN-3', 'RF']
    if nn_res: models_list.append('NN')
    
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(models_list))
    width = 0.25
    
    for i, task in enumerate(['country', 'time', 'activity']):
        vals = [all_results[m][task]['accuracy'] for m in models_list]
        ax.bar(x + (i-1)*width, vals, width, label=task.capitalize())
        
    ax.set_xticks(x)
    ax.set_xticklabels(models_list)
    ax.set_title(f'Model Performance - {feature_name}')
    ax.set_ylim(0, 1.0)
    ax.legend()
    plt.savefig(exp_fig_path / f'performance_comparison_{feature_name}.png')
    plt.close()
    
    # 2. Confusion Matrices (For best model, e.g., RF or NN)
    best_preds_c = pred_nn_c if nn_res else rf_c.predict(X_test_scaled)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(confusion_matrix(y_test_c, best_preds_c), ax=ax, cmap='Blues')
    ax.set_title(f'Country Confusion Matrix ({feature_name})')
    plt.savefig(exp_fig_path / f'confusion_matrix_country_{feature_name}.png')
    plt.close()
    
    # 3. TSNE (Subsampled for speed if needed, but here full test set)
    # Only if dims > 50
    if X_test_scaled.shape[1] > 50:
        tsne = TSNE(n_components=2, random_state=42, perplexity=30)
        X_emb = tsne.fit_transform(X_test_scaled)
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].scatter(X_emb[:,0], X_emb[:,1], c=y_test_c, cmap='tab20', s=10)
        axes[0].set_title('Country')
        axes[1].scatter(X_emb[:,0], X_emb[:,1], c=y_test_t, cmap='viridis', s=10)
        axes[1].set_title('Time')
        axes[2].scatter(X_emb[:,0], X_emb[:,1], c=y_test_a, cmap='Set1', s=10)
        axes[2].set_title('Activity')
        plt.savefig(exp_fig_path / f'tsne_clusters_{feature_name}.png')
        plt.close()

    # Save JSON Results for this feature set
    json_path = RESULTS_PATH / f'results_{feature_name}.json'
    with open(json_path, 'w') as f:
        json.dump(all_results, f, indent=2, default=float)
        
    print(f"   ✓ Saved results to {json_path}")
    print(f"   ✓ Saved figures to {exp_fig_path}")
    
    return all_results

## 4. Main Execution Loop
Iterate through all available feature sets and run the pipeline.

In [223]:
FEATURE_SETS = {
    'pretrained_original': 'places_features_pretrained.npy',
    'pretrained_improved': 'places_features_pretrained_improved.npy',
    'pretrained_normalized': 'places_features_pretrained_normalized.npy',
    'pretrained_minmax': 'places_features_pretrained_minmax.npy',
    'simple_original': 'places_features_simple.npy',
    'simple_normalized': 'places_features_simple_normalized.npy',
    'simple_minmax': 'places_features_simple_minmax.npy'
}

print("\n" + "="*80)
print("STARTING MULTI-FEATURE EVALUATION")
print("="*80)

final_summary = []

for name, file in FEATURE_SETS.items():
    res = run_experiment(name, file)
    if res:
        # Extract Neural Network (or RF) Country Accuracy as key metric
        best_acc = 0
        if 'NN' in res:
            best_acc = res['NN']['country']['accuracy']
        else:
            best_acc = res['RF']['country']['accuracy']
            
        final_summary.append({
            'Feature Set': name,
            'Best Country Acc': best_acc,
            'Best Time Acc': res['RF']['time']['accuracy'], # RF usually best for time
            'Best Activity Acc': res['NN']['activity']['accuracy'] if 'NN' in res else res['RF']['activity']['accuracy']
        })

print("\n" + "="*80)
print("FINAL SUMMARY COMPARISON")
print("="*80)
summary_df = pd.DataFrame(final_summary)
print(summary_df.sort_values('Best Country Acc', ascending=False).to_string(index=False))

# Save Overall Comparison
summary_df.to_csv(RESULTS_PATH / 'feature_comparison_summary.csv', index=False)


STARTING MULTI-FEATURE EVALUATION

################################################################################
🚀 RUNNING EXPERIMENT: PRETRAINED_ORIGINAL
################################################################################
   Loading places_features_pretrained.npy...
   Oversampling for Time task...
   Training KNN (k=3)...
   Training Random Forest...
   Training Neural Network...
   Generating Visualizations...
   ✓ Saved results to c:\Users\qadim\OneDrive - student.birzeit.edu\birzeit\5_year_1\machine_learning\assingment_3_project\travel-recommender-system\results\metrics\results_pretrained_original.json
   ✓ Saved figures to c:\Users\qadim\OneDrive - student.birzeit.edu\birzeit\5_year_1\machine_learning\assingment_3_project\travel-recommender-system\results\figures\pretrained_original

################################################################################
🚀 RUNNING EXPERIMENT: PRETRAINED_IMPROVED
###########################################################